# Sign Language Recognition Workflow

This notebook follows the same workflow as the earlier reference notebook:
1. confirm the environment and GPU
2. install the required packages into the selected kernel
3. set up MediaPipe hand extraction
4. prepare a sequence dataset
5. define the LSTM model
6. train and evaluate
7. save the final checkpoint

Use the `base (Python 3.10.21)` kernel which already has GPU support.

In [2]:
# Step 1: environment check
import sys
import torch

print(sys.executable)
print('Torch version:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
else:
    print('GPU: CPU fallback')

c:\Users\Janice\miniconda3\python.exe
Torch version: 2.7.0+cu126
CUDA available: True
GPU: NVIDIA GeForce RTX 4060 Ti


In [ ]:
# Step 2: install the project stack into the currently selected kernel
# Important: TensorFlow 2.15.1 is not compatible with the newer JAX/ml_dtypes stack.
# Remove that conflict first, then install the known-good versions for this notebook.
#%pip uninstall -y jax jaxlib ml-dtypes || true
#%pip install --upgrade --force-reinstall "ml-dtypes==0.3.2" "tensorflow==2.15.1" "opencv-contrib-python==4.11.0.86" "mediapipe==0.10.21" "protobuf<5" "pandas<3" "matplotlib" "imageio"

# print('Install complete.')

In [1]:
# Step 3: package sanity check
import cv2
import mediapipe as mp
import numpy as np
import torch

print('OpenCV:', cv2.__version__)
print('MediaPipe:', mp.__version__)
print('NumPy:', np.__version__)
print('Torch CUDA:', torch.cuda.is_available())
print('Has MediaPipe solutions:', hasattr(mp, 'solutions'))


OpenCV: 4.11.0
MediaPipe: 0.10.21
NumPy: 1.26.4
Torch CUDA: True
Has MediaPipe solutions: True


## Data and preprocessing flow

The model expects landmark sequences derived from image or video frames.

Typical flow:
- load frames or images from the dataset folders
- detect hands with MediaPipe
- flatten each detected hand into 3D landmark coordinates
- form a fixed-length sequence per sample
- map class names to numeric labels

## Model definition flow

The training step uses a sequence model to learn temporal patterns across frames.

This notebook keeps the structure simple:
- input dimension = landmark vector size
- LSTM layers capture temporal movement
- final classification head predicts the sign class

In [2]:
# Step 5: model definition
import sys
from pathlib import Path

PROJECT_ROOT = next(
    parent for parent in [Path.cwd(), *Path.cwd().parents]
    if (parent / 'models' / 'SX' / 'model.py').is_file()
)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from models.SX.model import HandSignLSTM

INPUT_SIZE = 63  # One hand: 21 landmarks * 3 coordinates
HIDDEN_SIZE = 64
NUM_LAYERS = 2
DROPOUT = 0.2

print('HandSignLSTM is ready for the dataset training step.')

HandSignLSTM is ready for the dataset training step.


## Training flow

This is the final stage of the workflow:
- build dataset loaders
- train the model
- track validation accuracy
- save the best checkpoint

In [3]:
# Step 6.1: preprocess images into cached landmark features
import importlib
import random

from torch.utils.data import DataLoader
import models.SX.train as sx_train
sx_train = importlib.reload(sx_train)
SignLanguageDataset = sx_train.SignLanguageDataset
cache_landmarks = sx_train.cache_landmarks
load_image_samples = sx_train.load_image_samples

SEED = 42
BATCH_SIZE = 32
SEQUENCE_LENGTH = 1  # The current dataset contains still images, not video clips.
TWO_HANDS = True
USE_BODY_POSE = True  # Set to True to test hands + body pose; False keeps the original baseline.
INPUT_SIZE = 225 if USE_BODY_POSE else 126
AUGMENT_TRAIN = True
AUGMENT_VALID = False # Set to True to augment the validation dataset as well.
AUGMENTATION_COPIES_PER_IMAGE = 2 # Set number of augmented images per original image.

try:
    from tensorflow.keras.preprocessing.image import ImageDataGenerator
except Exception as exc:
    ImageDataGenerator = None
    print(f'ImageDataGenerator unavailable: {exc}')


def create_augmented_dataset(source_dir, output_dir, copies_per_image=2):
    if ImageDataGenerator is None:
        raise RuntimeError(
            'ImageDataGenerator is unavailable. Install TensorFlow/Keras to enable augmentation.'
        )

    output_dir.mkdir(parents=True, exist_ok=True)
    datagen = ImageDataGenerator(
        rescale=1.0 / 255,
        shear_range=0.2,
        zoom_range=0.0,
        horizontal_flip=False,
    )

    created = 0
    for class_dir in sorted(source_dir.iterdir()):
        if not class_dir.is_dir():
            continue

        target_dir = output_dir / class_dir.name.lower()
        target_dir.mkdir(parents=True, exist_ok=True)

        for image_path in sorted(class_dir.iterdir()):
            if image_path.suffix.lower() not in {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}:
                continue

            image = cv2.imread(str(image_path), cv2.IMREAD_COLOR)
            if image is None:
                continue

            rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB).astype(np.float32)
            image_batch = np.expand_dims(rgb, axis=0)

            for index, batch in enumerate(datagen.flow(image_batch, batch_size=1, shuffle=False)):
                augmented_rgb = np.clip(batch[0] * 255.0, 0, 255).astype(np.uint8)
                augmented_bgr = cv2.cvtColor(augmented_rgb, cv2.COLOR_RGB2BGR)
                save_path = target_dir / f'{image_path.stem}_aug_{index + 1}.jpg'
                cv2.imwrite(str(save_path), augmented_bgr)
                created += 1

                if index + 1 >= copies_per_image:
                    break

    return created


random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
data_root = PROJECT_ROOT / 'SuXin_TEMP' / 'data'
cache_root = PROJECT_ROOT / 'SuXin_TEMP' / 'landmark_cache'
augmented_data_root = PROJECT_ROOT / 'SuXin_TEMP' / 'data_augmented'

if ImageDataGenerator is not None:
    train_aug_dir = augmented_data_root / 'train'
    valid_aug_dir = augmented_data_root / 'valid'

    if AUGMENT_TRAIN:
        train_generated = create_augmented_dataset(
            data_root / 'train',
            train_aug_dir,
            copies_per_image=AUGMENTATION_COPIES_PER_IMAGE,
        )
        print(f'Generated {train_generated} augmented training images in {train_aug_dir}')

    if AUGMENT_VALID:
        valid_generated = create_augmented_dataset(
            data_root / 'valid',
            valid_aug_dir,
            copies_per_image=max(1, AUGMENTATION_COPIES_PER_IMAGE // 2),
        )
        print(f'Generated {valid_generated} augmented validation images in {valid_aug_dir}')

    train_samples = load_image_samples(train_aug_dir if AUGMENT_TRAIN else data_root / 'train')
    valid_samples = load_image_samples(valid_aug_dir if AUGMENT_VALID else data_root / 'valid')
else:
    train_samples = load_image_samples(data_root / 'train')
    valid_samples = load_image_samples(data_root / 'valid')
    print('ImageDataGenerator unavailable; using the original dataset without augmentation.')

if not train_samples or not valid_samples:
    raise ValueError(f'No image samples found under {data_root}.')

class_names = sorted({label for _, label in train_samples + valid_samples})
class_to_idx = {name: index for index, name in enumerate(class_names)}

train_detected, train_missing = cache_landmarks(
    train_samples,
    cache_root / 'train',
    two_hands=TWO_HANDS,
    sequence_length=SEQUENCE_LENGTH,
    use_body_pose=USE_BODY_POSE,
)
valid_detected, valid_missing = cache_landmarks(
    valid_samples,
    cache_root / 'valid',
    two_hands=TWO_HANDS,
    sequence_length=SEQUENCE_LENGTH,
    use_body_pose=USE_BODY_POSE,
)

train_dataset = SignLanguageDataset(
    train_samples,
    class_to_idx,
    two_hands=TWO_HANDS,
    sequence_length=SEQUENCE_LENGTH,
    cache_dir=cache_root / 'train',
    use_body_pose=USE_BODY_POSE,
)
valid_dataset = SignLanguageDataset(
    valid_samples,
    class_to_idx,
    two_hands=TWO_HANDS,
    sequence_length=SEQUENCE_LENGTH,
    cache_dir=cache_root / 'valid',
    use_body_pose=USE_BODY_POSE,
)
train_loader = DataLoader(
    train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=0,
    pin_memory=device.type == 'cuda'
)
valid_loader = DataLoader(
    valid_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0,
    pin_memory=device.type == 'cuda'
)

print(f'Device: {device}')
print(f'Train samples: {len(train_dataset)} | Validation samples: {len(valid_dataset)}')
print(f'Classes ({len(class_names)}): {class_names}')
print(f'Train detections: {train_detected} | missing: {train_missing}')
print(f'Valid detections: {valid_detected} | missing: {valid_missing}')
print(f'Cached features: {cache_root}')

Generated 918 augmented training images in c:\Sean Su\NUS_Project\VisionSpeak\SuXin_TEMP\data_augmented\train
Device: cuda
Train samples: 918 | Validation samples: 45
Classes (18): ['apple', 'can', 'get', 'good', 'have', 'help', 'how', 'i', 'like', 'love', 'my', 'no', 'sorry', 'ty', 'want', 'yes', 'you', 'your']
Train detections: 909 | missing: 9
Valid detections: 45 | missing: 0
Cached features: c:\Sean Su\NUS_Project\VisionSpeak\SuXin_TEMP\landmark_cache


In [6]:
# Step 6.2: train the LSTM model on cached landmark features
EPOCHS = 200
LEARNING_RATE = 1e-3

import torch
import torch.nn as nn

from models.SX.model import HandSignLSTM

model = HandSignLSTM(
    input_size=INPUT_SIZE, hidden_size=HIDDEN_SIZE, num_layers=NUM_LAYERS,
    num_classes=len(class_names), dropout=DROPOUT
).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)
criterion = nn.CrossEntropyLoss()
checkpoint_filename = 'sx_hand_sign_cached_both_hands_body.pt' if USE_BODY_POSE else 'sx_hand_sign_cached_both_hands.pt'
checkpoint_path = PROJECT_ROOT / 'checkpoints' / checkpoint_filename
checkpoint_path.parent.mkdir(parents=True, exist_ok=True)
best_val_acc = 0.0

for epoch in range(1, EPOCHS + 1):
    model.train()
    train_loss = 0.0
    train_correct = 0
    train_total = 0

    for sequences, labels in train_loader:
        sequences = sequences.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)
        optimizer.zero_grad()
        logits = model(sequences)
        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()
        train_loss += loss.item() * labels.size(0)
        train_correct += (logits.argmax(dim=1) == labels).sum().item()
        train_total += labels.size(0)

    model.eval()
    valid_loss = 0.0
    valid_correct = 0
    valid_total = 0
    with torch.no_grad():
        for sequences, labels in valid_loader:
            sequences = sequences.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)
            logits = model(sequences)
            loss = criterion(logits, labels)
            valid_loss += loss.item() * labels.size(0)
            valid_correct += (logits.argmax(dim=1) == labels).sum().item()
            valid_total += labels.size(0)

    train_loss /= max(train_total, 1)
    valid_loss /= max(valid_total, 1)
    train_acc = train_correct / max(train_total, 1)
    valid_acc = valid_correct / max(valid_total, 1)
    print(
        f'Epoch {epoch:02d}/{EPOCHS} | '
        f'train loss {train_loss:.4f} acc {train_acc:.4f} | '
        f'valid loss {valid_loss:.4f} acc {valid_acc:.4f}'
    )

    if valid_acc > best_val_acc:
        best_val_acc = valid_acc
        torch.save(
            {
                'model_state_dict': model.state_dict(),
                'class_to_idx': class_to_idx,
                'idx_to_class': {index: name for name, index in class_to_idx.items()},
                'input_size': INPUT_SIZE,
                'hidden_size': HIDDEN_SIZE,
                'num_layers': NUM_LAYERS,
                'num_classes': len(class_names),
                'sequence_length': SEQUENCE_LENGTH,
                'two_hands': TWO_HANDS,
                'use_body_pose': USE_BODY_POSE,
                'best_val_acc': best_val_acc,
                'epoch': epoch,
            },
            checkpoint_path,
        )

        print(f'Saved best checkpoint: {checkpoint_path}')

print(f'Finished training. Best validation accuracy: {best_val_acc:.4f}')


Epoch 01/200 | train loss 2.8829 acc 0.0763 | valid loss 2.9055 acc 0.0222
Saved best checkpoint: c:\Sean Su\NUS_Project\VisionSpeak\checkpoints\sx_hand_sign_cached_both_hands_body.pt
Epoch 02/200 | train loss 2.8385 acc 0.0980 | valid loss 2.8927 acc 0.0444
Saved best checkpoint: c:\Sean Su\NUS_Project\VisionSpeak\checkpoints\sx_hand_sign_cached_both_hands_body.pt
Epoch 03/200 | train loss 2.7392 acc 0.1264 | valid loss 2.8353 acc 0.0222
Epoch 04/200 | train loss 2.6329 acc 0.1558 | valid loss 2.8407 acc 0.0222
Epoch 05/200 | train loss 2.5716 acc 0.1634 | valid loss 2.7664 acc 0.1111
Saved best checkpoint: c:\Sean Su\NUS_Project\VisionSpeak\checkpoints\sx_hand_sign_cached_both_hands_body.pt
Epoch 06/200 | train loss 2.5169 acc 0.1950 | valid loss 2.7462 acc 0.1111
Epoch 07/200 | train loss 2.4560 acc 0.2146 | valid loss 2.6711 acc 0.1333
Saved best checkpoint: c:\Sean Su\NUS_Project\VisionSpeak\checkpoints\sx_hand_sign_cached_both_hands_body.pt
Epoch 08/200 | train loss 2.3962 acc 0.

## Webcam prediction

Run the next cell after training. It loads the saved best checkpoint, reads webcam frames, extracts two-hand landmarks, runs the LSTM, and displays the predicted word. Press `q` in the webcam window to stop.

In [ ]:
# Step 8: real-time webcam prediction on macOS or Windows
import platform
import sys
from pathlib import Path

import cv2
import mediapipe as mp
import numpy as np
import torch

PROJECT_ROOT = next(
    parent for parent in [Path.cwd(), *Path.cwd().parents]
    if (parent / 'models' / 'SX' / 'model.py').is_file()
)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from models.SX.model import HandSignLSTM

body_checkpoint_path = PROJECT_ROOT / 'checkpoints' / 'sx_hand_sign_cached_both_hands_body.pt'
hand_checkpoint_path = PROJECT_ROOT / 'checkpoints' / 'sx_hand_sign_cached_both_hands.pt'

# Choose the checkpoint to use by commenting out the one you do not want.
checkpoint_path = body_checkpoint_path
# checkpoint_path = hand_checkpoint_path

if not checkpoint_path.is_file():
    raise FileNotFoundError(
        f'Checkpoint not found: {checkpoint_path}. Copy the trained checkpoint to the MacBook first.'
    )

if torch.backends.mps.is_available():
    device = torch.device('mps')
else:
    device = torch.device('cpu')

checkpoint = torch.load(checkpoint_path, map_location='cpu')
class_to_idx = checkpoint['class_to_idx']
idx_to_class = {index: name for name, index in class_to_idx.items()}

use_body_pose = checkpoint.get('use_body_pose', False)
two_hands = checkpoint.get('two_hands', True)
expected_input_size = checkpoint.get('input_size', 126 if two_hands else 63)

model = HandSignLSTM(
    input_size=expected_input_size,
    hidden_size=checkpoint.get('hidden_size', 64),
    num_layers=checkpoint.get('num_layers', 2),
    num_classes=checkpoint.get('num_classes', len(class_to_idx)),
).to(device)
model.load_state_dict(checkpoint['model_state_dict'])
model.eval()

mp_hands = mp.solutions.hands
mp_pose = mp.solutions.pose
mp_drawing = mp.solutions.drawing_utils
window_name = 'VisionSpeak webcam prediction'

if platform.system() == 'Darwin':
    camera = cv2.VideoCapture(0, cv2.CAP_AVFOUNDATION)
else:
    camera = cv2.VideoCapture(0)

if not camera.isOpened():
    raise RuntimeError(
        'Could not open the webcam. On macOS, allow camera access for VS Code or Terminal in System Settings.'
    )

cv2.startWindowThread()
cv2.namedWindow(window_name, cv2.WINDOW_NORMAL)
cv2.moveWindow(window_name, 20, 20)
print(f'Using device: {device}')
print(f'Checkpoint feature layout: {expected_input_size} dims (two_hands={two_hands}, use_body_pose={use_body_pose})')
print('Press q or click the window close button to stop.')

hand_feature_dim = 126 if two_hands else 63
body_feature_dim = 99 if use_body_pose else 0

try:
    with mp_hands.Hands(
        static_image_mode=False,
        max_num_hands=2,
        min_detection_confidence=0.5,
        min_tracking_confidence=0.5,
    ) as hands, mp_pose.Pose(
        static_image_mode=True,
        min_detection_confidence=0.5,
        min_tracking_confidence=0.5,
    ) as pose:
        while True:
            success, frame = camera.read()
            if not success:
                print('Could not read a webcam frame.')
                break

            frame = cv2.flip(frame, 1)
            rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            results = hands.process(rgb)
            pose_results = pose.process(rgb)
            prediction_text = 'No hand detected'

            if results.multi_hand_landmarks:
                hand_vectors = []
                frame_h, frame_w = frame.shape[:2]
                for hand_landmarks in results.multi_hand_landmarks[:2]:
                    coordinates = []
                    xs = []
                    ys = []
                    for landmark in hand_landmarks.landmark:
                        coordinates.extend([landmark.x, landmark.y, landmark.z])
                        xs.append(landmark.x * frame_w)
                        ys.append(landmark.y * frame_h)
                    hand_vectors.append(np.asarray(coordinates, dtype=np.float32))
                    mp_drawing.draw_landmarks(frame, hand_landmarks, mp_hands.HAND_CONNECTIONS)

                    x1 = max(0, int(min(xs)))
                    y1 = max(0, int(min(ys)))
                    x2 = min(frame_w - 1, int(max(xs)))
                    y2 = min(frame_h - 1, int(max(ys)))
                    cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 255, 255), 2)

                while len(hand_vectors) < 2 and two_hands:
                    hand_vectors.append(np.zeros(63, dtype=np.float32))

                if two_hands:
                    hand_features = np.concatenate(hand_vectors[:2]).astype(np.float32)
                else:
                    hand_features = hand_vectors[0].astype(np.float32) if hand_vectors else np.zeros(63, dtype=np.float32)

                hand_features = hand_features[:hand_feature_dim]
                if hand_features.shape[0] < hand_feature_dim:
                    hand_features = np.concatenate([
                        hand_features,
                        np.zeros(hand_feature_dim - hand_features.shape[0], dtype=np.float32),
                    ])

                if use_body_pose:
                    body_vector = np.zeros(body_feature_dim, dtype=np.float32)
                    if pose_results.pose_landmarks:
                        coordinates = []
                        for landmark in pose_results.pose_landmarks.landmark:
                            coordinates.extend([landmark.x, landmark.y, landmark.z])
                        body_vector = np.asarray(coordinates[:body_feature_dim], dtype=np.float32)
                        if body_vector.shape[0] < body_feature_dim:
                            body_vector = np.concatenate([
                                body_vector,
                                np.zeros(body_feature_dim - body_vector.shape[0], dtype=np.float32),
                            ])
                    features = np.concatenate([hand_features, body_vector]).astype(np.float32)
                else:
                    features = hand_features

                input_tensor = torch.from_numpy(features).view(1, 1, expected_input_size).to(device)

                with torch.no_grad():
                    probabilities = torch.softmax(model(input_tensor), dim=1)
                    confidence, prediction_index = probabilities.max(dim=1)
                word = idx_to_class[int(prediction_index.item())]
                prediction_text = f'{word} ({confidence.item():.1%})'

            cv2.putText(
                frame, prediction_text, (20, 45), cv2.FONT_HERSHEY_SIMPLEX,
                1.0, (0, 255, 0), 2, cv2.LINE_AA
            )
            cv2.imshow(window_name, frame)
            key = cv2.waitKey(1) & 0xFF
            if key in (ord('q'), 27):
                break
finally:
    try:
        cv2.destroyWindow(window_name)
    except cv2.error:
        pass
    camera.release()
    cv2.destroyAllWindows()

print('Webcam prediction stopped.')

2026-09-12 00:02:17.291 python[14280:115608] WARNING: AVCaptureDeviceTypeExternal is deprecated for Continuity Cameras. Please use AVCaptureDeviceTypeContinuityCamera and add NSCameraUseContinuityCameraDeviceType to your Info.plist.


Using device: mps
Checkpoint feature layout: 225 dims (two_hands=True, use_body_pose=True)
Press q or click the window close button to stop.


I0000 00:00:1789142538.058062  115608 gl_context.cc:357] GL version: 2.1 (2.1 Metal - 89.4), renderer: Apple M4
I0000 00:00:1789142538.069478  115608 gl_context.cc:357] GL version: 2.1 (2.1 Metal - 89.4), renderer: Apple M4
INFO: Created TensorFlow Lite XNNPACK delegate for CPU.
W0000 00:00:1789142538.094762  118616 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1789142538.106442  118616 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1789142538.234985  118629 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1789142538.248531  118637 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback t